In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(model=model, lr_init=1e-3)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.24306529760360718
epoch:  1, loss: 0.24175114929676056
epoch:  2, loss: 0.24044565856456757
epoch:  3, loss: 0.23914878070354462
epoch:  4, loss: 0.23786041140556335
epoch:  5, loss: 0.2365804761648178
epoch:  6, loss: 0.2353089451789856
epoch:  7, loss: 0.23404574394226074
epoch:  8, loss: 0.23279082775115967
epoch:  9, loss: 0.2315441519021988
epoch:  10, loss: 0.23030564188957214
epoch:  11, loss: 0.22907523810863495
epoch:  12, loss: 0.22785285115242004
epoch:  13, loss: 0.22663845121860504
epoch:  14, loss: 0.22543199360370636
epoch:  15, loss: 0.22423338890075684
epoch:  16, loss: 0.2230425924062729
epoch:  17, loss: 0.22185954451560974
epoch:  18, loss: 0.220684215426445
epoch:  19, loss: 0.2195165604352951
epoch:  20, loss: 0.21835650503635406
epoch:  21, loss: 0.21720410883426666
epoch:  22, loss: 0.21605916321277618
epoch:  23, loss: 0.21492168307304382
epoch:  24, loss: 0.21379154920578003
epoch:  25, loss: 0.2126687616109848
epoch:  26, loss: 0.2115532308

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -2842.564697265625
Test metrics:  R2 = -3355.05615234375


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3138257563114166
epoch:  1, loss: 0.17336417734622955
epoch:  2, loss: 0.10409080237150192
epoch:  3, loss: 0.07119204103946686
epoch:  4, loss: 0.054174814373254776
epoch:  5, loss: 0.04483594745397568
epoch:  6, loss: 0.03979749232530594
epoch:  7, loss: 0.037123098969459534
epoch:  8, loss: 0.035723790526390076
epoch:  9, loss: 0.03499879688024521
epoch:  10, loss: 0.03462596237659454
epoch:  11, loss: 0.03443474322557449
epoch:  12, loss: 0.03433652222156525
epoch:  13, loss: 0.03428574278950691
epoch:  14, loss: 0.03425902873277664
epoch:  15, loss: 0.034244537353515625
epoch:  16, loss: 0.03423617407679558
epoch:  17, loss: 0.034230995923280716
epoch:  18, loss: 0.034200385212898254
epoch:  19, loss: 0.03419274091720581
epoch:  20, loss: 0.03417111933231354
epoch:  21, loss: 0.03415527194738388
epoch:  22, loss: 0.03414031118154526
epoch:  23, loss: 0.03412807360291481
epoch:  24, loss: 0.03410881757736206
epoch:  25, loss: 0.03408726677298546
epoch:  26, loss:

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7220671772956848
Test metrics:  R2 = 0.6787985563278198
